In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rdkit torch-geometric py3dmol joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.6 MB/s eta 0:00:00


In [3]:
!pip install rdkit torch-geometric biopython py3Dmol xgboost scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.7 MB/s eta 0:00:00


In [4]:
!apt-get -qq install openbabel

Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previously unselected package openbabel.
Preparing to unpack .../openbabel_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking openbabel (3.1.1+dfsg-6ubuntu5) ...
Setting up libboost-iostreams1.74.0:amd64 (

In [5]:
import os, time, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
import numpy as np
from pathlib import Path
from google.colab import drive
from IPython.display import display, HTML

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdFingerprintGenerator
from rdkit.Chem import rdMolAlign, rdMolTransforms
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool
from Bio.PDB import PDBParser
import py3Dmol

# ── 1. PATHS ─────────────────────────────────────────────────────────────────
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE            = "/content/drive/MyDrive/PredictionPipeline"
CACHE_DIR       = Path("/content/drive/MyDrive/thesis_cache")
GRAPH_CACHE     = CACHE_DIR / "graphs_cache.pkl"
LIGAND_DIR      = "/content/drive/MyDrive/102kOutFiles"   # _out.pdbqt klasörü

GNN_MODEL_PATH  = f"{BASE}/gnn_model_final.pt"
XGB_MODEL_PATH  = f"{BASE}/xgb_model_final.joblib"
XGB_SCALER_PATH = f"{BASE}/xgb_scaler.joblib"
RECEPTOR_PDBQT  = f"{BASE}/receptor.pdbqt"
LIGAND_SDF      = f"{BASE}/ZINC000002622269.sdf"   # tahmin edilecek ligand

ACTIVE_THRESHOLD = -7.0
GNN_WEIGHT       = 0.6
LIGAND_SUFFIX    = "_out.pdbqt"

DOCKING_CENTER   = (-62.593, -54.158, -53.827)
DOCKING_SIZE     = (60.0, 66.0, 90.0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── 2. GNN ARCHITECTURE (eğitimle aynı) ─────────────────────────────────────
class ProteinLigandGAT(nn.Module):
    def __init__(self, node_dim=18, edge_dim=2, hidden=128, heads=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(node_dim, hidden)
        self.conv1 = GATConv(hidden, hidden//heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
        self.conv2 = GATConv(hidden, hidden//heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
        self.conv3 = GATConv(hidden, hidden//heads, heads=heads, edge_dim=edge_dim, dropout=dropout)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(hidden)
        self.bn3   = nn.BatchNorm1d(hidden)
        self.dropout = nn.Dropout(dropout)
        self.fc1   = nn.Linear(hidden, 64)
        self.fc2   = nn.Linear(64, 32)
        self.fc3   = nn.Linear(32, 1)

    def forward(self, data):
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = F.relu(self.input_proj(x))
        x = self.bn1(F.elu(self.conv1(x, ei, ea)) + x); x = self.dropout(x)
        x = self.bn2(F.elu(self.conv2(x, ei, ea)) + x); x = self.dropout(x)
        x = self.bn3(F.elu(self.conv3(x, ei, ea)) + x); x = self.dropout(x)
        x = global_mean_pool(x, b)
        x = F.relu(self.fc1(x)); x = self.dropout(x)
        x = F.relu(self.fc2(x)); x = self.dropout(x)
        return self.fc3(x).squeeze(-1)

    def get_embedding(self, data):
        """
        Return the 128-dim graph embedding (before final FC layers).
        Used for similarity-based pose retrieval.
        """
        x, ei, ea, b = data.x, data.edge_index, data.edge_attr, data.batch
        x = F.relu(self.input_proj(x))
        x = self.bn1(F.elu(self.conv1(x, ei, ea)) + x); x = self.dropout(x)
        x = self.bn2(F.elu(self.conv2(x, ei, ea)) + x); x = self.dropout(x)
        x = self.bn3(F.elu(self.conv3(x, ei, ea)) + x); x = self.dropout(x)
        return global_mean_pool(x, b)   # (1, 128)

# ── 3. GRAPH HELPERS ─────────────────────────────────────────────────────────
ATOM_TYPES = [6, 7, 8, 9, 15, 16, 17, 35, 53]

def get_coords(mol):
    c = mol.GetConformer()
    return np.array([list(c.GetAtomPosition(i))
                     for i in range(mol.GetNumAtoms())])

def atom_features(atom, is_protein=False):
    oh = [int(atom.GetAtomicNum() == t) for t in ATOM_TYPES]
    oh.append(int(atom.GetAtomicNum() not in ATOM_TYPES))
    return oh + [
        atom.GetTotalValence()/8., atom.GetFormalCharge()/2.,
        float(atom.GetIsAromatic()), float(atom.IsInRing()),
        float(atom.GetTotalNumHs()>0),
        float(atom.GetAtomicNum() in [7,8]),
        float(is_protein), 0.
    ]

def get_receptor_pdb():
    import subprocess
    p = f"{BASE}/receptor.pdb"
    if not os.path.exists(p):
        print("  Converting PDBQT → PDB via obabel...")
        subprocess.run(["obabel", RECEPTOR_PDBQT, "-O", p, "-h"],
                       capture_output=True)
    return p

def extract_pocket_for_graph(lig_coords, cutoff=10.0):
    parser    = PDBParser(QUIET=True)
    structure = parser.get_structure("rec", get_receptor_pdb())
    pc, pf    = [], []
    for model in structure:
        for chain in model:
            for res in chain:
                rc, rf = [], []
                for atom in res:
                    c = np.array(atom.get_vector().get_array())
                    f = [0]*len(ATOM_TYPES)+[0,0.,0.,0.,0.,0.,0.,1.,0.]
                    rc.append(c); rf.append(f)
                if rc:
                    md = min(np.linalg.norm(lig_coords-c,axis=1).min()
                             for c in rc)
                    if md <= cutoff:
                        pc.extend(rc); pf.extend(rf)
    return np.array(pc), np.array(pf)

def build_graph_from_mol(mol):
    """Build graph the same way training data was built."""
    lc = get_coords(mol)
    pc, pf = extract_pocket_for_graph(lc)
    lf = [atom_features(a, False) for a in mol.GetAtoms()]
    nl = len(lf)
    x  = torch.tensor(lf+list(pf), dtype=torch.float)
    es, ed, ea = [], [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        d   = float(np.linalg.norm(lc[i]-lc[j]))
        for s,t in [(i,j),(j,i)]:
            es.append(s); ed.append(t); ea.append([d,0.])
    for li in range(nl):
        for pi in range(len(pc)):
            d = float(np.linalg.norm(lc[li]-pc[pi]))
            if d<=5.:
                pg=pi+nl
                for s,t in [(li,pg),(pg,li)]:
                    es.append(s); ed.append(t); ea.append([d,1.])
    if not es: return None
    return Data(x=x,
                edge_index=torch.tensor([es,ed],dtype=torch.long),
                edge_attr =torch.tensor(ea,     dtype=torch.float))

# ── 4. POSE RETRIEVAL ENGINE ─────────────────────────────────────────────────
class PoseRetrievalEngine:
    """
    GNN Embedding Similarity → Nearest-Neighbor Pose Retrieval.

    How it works
    ────────────
    Training graphs were built from real Vina-docked poses (_out.pdbqt).
    Each graph encodes both ligand chemistry AND protein environment.

    For a new ligand:
      1. Build its graph (ligand centered at docking box, same as training).
      2. Extract its GNN embedding (128-dim global mean pool).
      3. Compute cosine similarity with ALL training embeddings.
      4. Retrieve top-K most similar training ligands' _out.pdbqt files.
      5. Read the best Vina pose from each file.
      6. Align the new ligand's 3D shape to the retrieved pose via
         heavy-atom RMS fitting (MCS-based) → ligand-specific pose.

    Result: each ligand gets a unique pose grounded in real docking data,
    not a generic translation to docking center.
    """

    def __init__(self, gnn_model, top_k=5):
        self.gnn   = gnn_model
        self.top_k = top_k
        self._emb_cache = None   # (N, 128) tensor built lazily

        print("\nLoading training graph cache...")
        t0 = time.time()
        with open(GRAPH_CACHE, "rb") as f:
            cached = pickle.load(f)

        self.train_graphs     = cached["graphs"]      # list of Data
        self.train_labels     = np.array(cached["labels"])
        self.train_filenames  = cached["file_names"]  # list of str

        print(f"  Loaded {len(self.train_graphs):,} training graphs "
              f"in {time.time()-t0:.1f}s")

    # ── embedding index ───────────────────────────────────────────────────
    def _build_embedding_index(self):
        """
        Compute GNN embeddings for ALL training graphs.
        Done once, then cached in RAM.
        ~2-5 min for 100K graphs on T4.
        """
        print("\n  Building embedding index from training graphs...")
        print("  (This runs once per session — ~3-5 min for 100K graphs)")
        t0     = time.time()
        embs   = []
        BATCH  = 256

        self.gnn.eval()
        with torch.no_grad():
            for i in range(0, len(self.train_graphs), BATCH):
                batch_graphs = self.train_graphs[i:i+BATCH]
                batch        = Batch.from_data_list(
                    [g.to(DEVICE) for g in batch_graphs])
                emb = self.gnn.get_embedding(batch)   # (B, 128)
                embs.append(emb.cpu())

                if (i // BATCH) % 50 == 0:
                    pct = i / len(self.train_graphs) * 100
                    elapsed = time.time()-t0
                    eta     = elapsed/max(i,1)*len(self.train_graphs) - elapsed
                    print(f"    {pct:.1f}%  elapsed={elapsed:.0f}s  "
                          f"ETA={eta:.0f}s", end='\r')

        self._emb_cache = torch.cat(embs, dim=0)   # (N, 128)
        # L2-normalise for cosine similarity via dot product
        norms = self._emb_cache.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self._emb_cache = self._emb_cache / norms
        print(f"\n  Index built: {self._emb_cache.shape}  "
              f"({time.time()-t0:.1f}s total)")

    def _get_query_embedding(self, query_graph):
        """Extract and L2-normalise embedding for one query graph."""
        self.gnn.eval()
        with torch.no_grad():
            batch = Batch.from_data_list([query_graph.to(DEVICE)])
            emb   = self.gnn.get_embedding(batch).cpu()   # (1, 128)
        emb = emb / emb.norm(dim=1, keepdim=True).clamp(min=1e-8)
        return emb   # (1, 128)

    # ── pose reading ──────────────────────────────────────────────────────
    def _read_best_pose_from_pdbqt(self, filename):
        """
        Read MODEL 1 (best Vina pose) from _out.pdbqt.
        Returns np.ndarray (N_heavy, 3) of heavy-atom coordinates.
        """
        # Try to find the file
        candidates = [
            os.path.join(LIGAND_DIR, filename + LIGAND_SUFFIX),
            os.path.join(LIGAND_DIR, filename),
            os.path.join(LIGAND_DIR, Path(filename).stem + LIGAND_SUFFIX),
        ]
        path = next((p for p in candidates if os.path.exists(p)), None)
        if path is None:
            return None

        coords = []
        in_model1  = False
        model_seen = 0
        with open(path) as f:
            for line in f:
                if line.startswith("MODEL"):
                    model_seen += 1
                    in_model1 = (model_seen == 1)
                    continue
                if line.startswith("ENDMDL") and in_model1:
                    break
                take = (model_seen == 0) or in_model1
                if take and line.startswith(("ATOM","HETATM")):
                    try:
                        x = float(line[30:38])
                        y = float(line[38:46])
                        z = float(line[46:54])
                        elem = line[76:78].strip() if len(line)>76 else 'C'
                        if elem.upper() != 'H':   # heavy atoms only
                            coords.append([x,y,z])
                    except ValueError:
                        pass
        return np.array(coords) if coords else None

    # ── shape-guided alignment ────────────────────────────────────────────
    def _align_ligand_to_pose(self, query_mol, ref_coords):
        """
        Align query_mol's 3D conformation to the retrieved pose coordinates.

        Strategy:
          1. Translate query centroid → ref centroid.
          2. Apply SVD-based rotation to minimise heavy-atom RMSD
             (works even if atom counts differ — uses centroid + inertia
             tensor alignment, not MCS).

        Returns a new RDKit Mol with updated conformer.
        """
        q_coords = get_coords(query_mol)          # (N_q, 3)
        r_coords = ref_coords                     # (N_r, 3)

        q_cen = q_coords.mean(axis=0)
        r_cen = r_coords.mean(axis=0)

        # Step 1: translate
        q_centered = q_coords - q_cen
        r_centered = r_coords - r_cen

        # Step 2: SVD rotation — align principal axes
        # Use the inertia-tensor approach (works regardless of atom count)
        # Build cross-covariance from common number of atoms
        n = min(len(q_centered), len(r_centered))
        H = q_centered[:n].T @ r_centered[:n]
        U, S, Vt = np.linalg.svd(H)
        # Correct for reflection
        d = np.linalg.det(Vt.T @ U.T)
        D = np.diag([1., 1., d])
        rot = Vt.T @ D @ U.T                    # (3, 3)

        # Apply rotation + translation
        q_new = (q_centered @ rot.T) + r_cen

        # Write new coords into a copy of the mol
        rw   = Chem.RWMol(query_mol)
        conf = rw.GetConformer()
        for i in range(rw.GetNumAtoms()):
            conf.SetAtomPosition(i, q_new[i].tolist())

        return rw.GetMol(), float(np.sqrt(
            ((q_new[:n] - r_centered[:n] + r_cen)**2).mean()))

    # ── public API ────────────────────────────────────────────────────────
    def retrieve_pose(self, query_mol, query_graph):
        """
        Main entry point.

        Parameters
        ----------
        query_mol   : RDKit Mol with 3D conformer (from ZINC SDF)
        query_graph : torch_geometric Data built from query_mol

        Returns
        -------
        posed_mol   : RDKit Mol with retrieved + aligned pose
        meta        : dict with similarity, source filename, rmsd info
        """
        # Build index if not done yet
        if self._emb_cache is None:
            self._build_embedding_index()

        # Query embedding
        print("\n  Computing query embedding...")
        q_emb = self._get_query_embedding(query_graph)   # (1, 128)

        # Cosine similarities
        sims  = (self._emb_cache @ q_emb.T).squeeze(1)   # (N,)
        top_k_idx = torch.topk(sims, self.top_k).indices.numpy()

        print(f"  Top-{self.top_k} similar training ligands:")
        best_mol   = None
        best_sim   = -1
        best_meta  = {}

        for rank, idx in enumerate(top_k_idx):
            fname = self.train_filenames[idx]
            sim   = float(sims[idx])
            label = float(self.train_labels[idx])

            ref_coords = self._read_best_pose_from_pdbqt(fname)
            if ref_coords is None:
                print(f"    [{rank+1}] {fname}  sim={sim:.4f}  "
                      f"label={label:.2f}  ⚠ file not found")
                continue

            posed_mol, rmsd = self._align_ligand_to_pose(query_mol, ref_coords)
            print(f"    [{rank+1}] {fname}  sim={sim:.4f}  "
                  f"label={label:.2f} kcal/mol  RMSD={rmsd:.2f} Å")

            if sim > best_sim:
                best_sim  = sim
                best_mol  = posed_mol
                best_meta = {
                    "source_file" : fname,
                    "similarity"  : sim,
                    "ref_label"   : label,
                    "align_rmsd"  : rmsd,
                    "rank"        : rank+1,
                }

        if best_mol is None:
            print("  WARNING: No _out.pdbqt files found. "
                  "Falling back to docking center translation.")
            best_mol  = _translate_to_center(query_mol)
            best_meta = {"source_file": "fallback", "similarity": 0.}

        fc = get_coords(best_mol).mean(axis=0)
        print(f"\n  Final pose centroid : "
              f"({fc[0]:.2f}, {fc[1]:.2f}, {fc[2]:.2f})")
        print(f"  Best similarity     : {best_meta['similarity']:.4f}")
        return best_mol, best_meta

def _translate_to_center(mol):
    """Fallback: translate ligand centroid to docking center."""
    coords = get_coords(mol)
    t      = np.array(DOCKING_CENTER) - coords.mean(axis=0)
    rw     = Chem.RWMol(mol)
    cf     = rw.GetConformer()
    for i in range(rw.GetNumAtoms()):
        p = cf.GetAtomPosition(i)
        cf.SetAtomPosition(i, (p.x+t[0], p.y+t[1], p.z+t[2]))
    return rw.GetMol()

# ── 5. PREDICTOR ─────────────────────────────────────────────────────────────
class BioVanguardPredictor:
    def __init__(self):
        print("Loading BioVanguard models...")
        self.device = DEVICE
        self.mfpgen = rdFingerprintGenerator.GetMorganGenerator(
            radius=2, fpSize=2048)

        self.gnn = ProteinLigandGAT().to(self.device)
        self.gnn.load_state_dict(
            torch.load(GNN_MODEL_PATH, map_location=self.device))
        self.gnn.eval()
        print("  [OK] GNN loaded.")

        self.xgb    = joblib.load(XGB_MODEL_PATH)
        self.scaler = joblib.load(XGB_SCALER_PATH)
        print("  [OK] XGBoost + scaler loaded.")

    def predict(self, mol):
        """Ensemble prediction — unchanged from working version."""
        fp   = np.array(self.mfpgen.GetFingerprint(mol))
        desc = np.array([
            Descriptors.MolWt(mol),         Descriptors.MolLogP(mol),
            Descriptors.NumHAcceptors(mol), Descriptors.NumHDonors(mol),
            Descriptors.NumRotatableBonds(mol), Descriptors.TPSA(mol),
            Descriptors.NumAromaticRings(mol), Descriptors.HeavyAtomCount(mol),
            Descriptors.FractionCSP3(mol),
        ], dtype=float)
        xf          = np.concatenate([fp,desc]).reshape(1,-1)
        xf[:,2048:] = self.scaler.transform(xf[:,2048:])
        xpred       = float(self.xgb.predict(xf)[0])
        print(f"  XGBoost : {xpred:.3f} kcal/mol")

        graph = build_graph_from_mol(mol)
        if graph is None:
            print("  WARNING: empty graph"); return xpred, None

        batch = Batch.from_data_list([graph]).to(self.device)
        with torch.no_grad():
            gpred = float(self.gnn(batch).cpu().numpy()[0])
        print(f"  GNN     : {gpred:.3f} kcal/mol")
        ens = GNN_WEIGHT*gpred + (1-GNN_WEIGHT)*xpred
        print(f"  Ensemble: {ens:.3f} kcal/mol")
        return ens, graph

# ── 6. VISUALIZATION ─────────────────────────────────────────────────────────
def clean_pdbqt(path):
    lines = []
    with open(path) as f:
        for line in f:
            if line.startswith(('ATOM','HETATM','TER','END')):
                lines.append(line[:66].rstrip()+'\n')
    r = ''.join(lines)
    if not r.strip(): raise RuntimeError(f"Empty PDBQT: {path}")
    return r

def build_box_edges(center, size):
    cx,cy,cz = center
    hx,hy,hz = size[0]/2,size[1]/2,size[2]/2
    C = [(cx-hx,cy-hy,cz-hz),(cx+hx,cy-hy,cz-hz),
         (cx+hx,cy+hy,cz-hz),(cx-hx,cy+hy,cz-hz),
         (cx-hx,cy-hy,cz+hz),(cx+hx,cy-hy,cz+hz),
         (cx+hx,cy+hy,cz+hz),(cx-hx,cy+hy,cz+hz)]
    E = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),
         (0,4),(1,5),(2,6),(3,7)]
    return C,E

def visualize_docking(posed_mol, ens_score=None, pose_meta=None,
                      pocket_cutoff=6., surface_cutoff=9.,
                      show_box=True, width=920, height=680):
    print("\nRendering 3D viewer...")
    view     = py3Dmol.view(width=width, height=height)
    coords   = get_coords(posed_mol)
    cx,cy,cz = coords.mean(axis=0).tolist()

    # Ligand
    view.addModel(Chem.MolToMolBlock(posed_mol), 'mol')
    view.setStyle({'model':0},{
        'stick': {'colorscheme':'greenCarbon','radius':0.20},
        'sphere':{'colorscheme':'greenCarbon','radius':0.30}})

    # Receptor
    rec = clean_pdbqt(RECEPTOR_PDBQT)
    view.addModel(rec,'pdb')
    view.setStyle({'model':1},{'cartoon':{'color':'spectrum','opacity':0.40}})

    # Binding pocket residues
    view.addStyle(
        {'model':1,'within':{'distance':pocket_cutoff,'sel':{'model':0}}},
        {'stick':{'color':'#FF8C00','radius':0.16},
         'cartoon':{'color':'#FF8C00','opacity':0.95}})

    # Pocket VDW surface
    view.addSurface(py3Dmol.VDW,
        {'opacity':0.18,'colorscheme':'whiteCarbon'},
        {'model':1,'within':{'distance':surface_cutoff,'sel':{'model':0}}})

    # Docking box
    if show_box:
        C,E = build_box_edges(DOCKING_CENTER, DOCKING_SIZE)
        for i,j in E:
            view.addCylinder({
                'start':{'x':C[i][0],'y':C[i][1],'z':C[i][2]},
                'end':  {'x':C[j][0],'y':C[j][1],'z':C[j][2]},
                'radius':0.22,'color':'cyan','opacity':0.50})

    # Labels
    base = dict(backgroundColor='#111111',backgroundOpacity=0.72,
                fontColor='white',fontSize=13,borderThickness=0,inFront=True)

    view.addLabel("Predicted binding pocket",
                  {**base,'fontColor':'#FF8C00',
                   'position':{'x':cx,'y':cy+5.,'z':cz}})

    if ens_score is not None:
        col = '#00E676' if ens_score<=ACTIVE_THRESHOLD else '#FF5252'
        act = 'Active'  if ens_score<=ACTIVE_THRESHOLD else 'Inactive'
        view.addLabel(
            f"Predicted ΔG = {ens_score:.2f} kcal/mol  |  {act}",
            {**base,'fontColor':col,'position':{'x':cx,'y':cy+9.,'z':cz}})

    if pose_meta:
        sim  = pose_meta.get('similarity', 0)
        src  = Path(pose_meta.get('source_file','')).stem
        rmsd = pose_meta.get('align_rmsd', 0)
        view.addLabel(
            f"Template: {src}  |  sim={sim:.3f}  |  RMSD={rmsd:.1f}Å",
            {**base,'fontColor':'#80CFFF',
             'position':{'x':cx,'y':cy+13.,'z':cz}})

    view.zoomTo({'model':0}); view.zoom(0.70)
    view.render()
    display(HTML(f"""
        <div style="border:1px solid #2a2a3e;border-radius:10px;
                    overflow:hidden;background:#0d0d1a;">
          <div style="font-family:monospace;font-size:11px;color:#7a7a9a;
                      padding:5px 10px;border-bottom:1px solid #2a2a3e;
                      background:#12122a;">
            BioVanguard — GNN Embedding-Based Pose Retrieval
            &nbsp;|&nbsp;Rotate:drag &nbsp;|&nbsp;Zoom:scroll
            &nbsp;|&nbsp;Pan:right-drag
            &nbsp;|&nbsp;<span style="color:#FF8C00">■</span> binding pocket
            &nbsp;|&nbsp;<span style="color:#00cc44">■</span> ligand
            &nbsp;|&nbsp;<span style="color:cyan">■</span> search box
          </div>
          {view._make_html()}
        </div>"""))
    print("  Viewer rendered.")

# ── 7. MAIN ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n"+"="*60)
    print("  BioVanguard — GNN Similarity-Based Pose Retrieval")
    print("  No Vina required at inference time")
    print("="*60)

    # Step 1: Load ligand
    print(f"\n[1/4] Loading ligand: {LIGAND_SDF}")
    mol = Chem.MolFromMolFile(LIGAND_SDF, removeHs=True, sanitize=True)
    if mol is None: raise ValueError("Cannot read SDF")
    if not mol.GetNumConformers(): raise ValueError("No 3D conformer in SDF")
    print(f"  Atoms: {mol.GetNumAtoms()}  Bonds: {mol.GetNumBonds()}")

    # Step 2: Affinity prediction (uses original SDF geometry — unchanged)
    print("\n[2/4] Affinity prediction (GNN + XGBoost)...")
    predictor         = BioVanguardPredictor()
    score, query_graph = predictor.predict(mol)

    activity = ("STRONG BINDING (Active)"
                if score<=ACTIVE_THRESHOLD else "WEAK BINDING (Inactive)")
    print("\n"+"="*60)
    print(f"  ENSEMBLE SCORE : {score:.3f} kcal/mol")
    print(f"  STATUS         : {activity}")
    print("="*60)

    # Step 3: Pose retrieval via GNN embedding similarity
    print("\n[3/4] Retrieving pose from training data...")
    if query_graph is None:
        print("  WARNING: No graph — using docking center fallback.")
        posed_mol = _translate_to_center(mol)
        pose_meta = {"source_file": "fallback", "similarity": 0.}
    else:
        pose_engine        = PoseRetrievalEngine(predictor.gnn, top_k=5)
        posed_mol, pose_meta = pose_engine.retrieve_pose(mol, query_graph)

    # Step 4: Visualize
    print("\n[4/4] Visualization...")
    visualize_docking(
        posed_mol      = posed_mol,
        ens_score      = score,
        pose_meta      = pose_meta,
        pocket_cutoff  = 6.,
        surface_cutoff = 9.,
        show_box       = True,
        width=920, height=680)

Device: cuda

  BioVanguard — GNN Similarity-Based Pose Retrieval
  No Vina required at inference time

[1/4] Loading ligand: /content/drive/MyDrive/PredictionPipeline/ZINC000002622269.sdf
  Atoms: 28  Bonds: 31

[2/4] Affinity prediction (GNN + XGBoost)...
Loading BioVanguard models...
  [OK] GNN loaded.
  [OK] XGBoost + scaler loaded.
  XGBoost : -7.299 kcal/mol
  GNN     : -7.356 kcal/mol
  Ensemble: -7.333 kcal/mol

  ENSEMBLE SCORE : -7.333 kcal/mol
  STATUS         : STRONG BINDING (Active)

[3/4] Retrieving pose from training data...

Loading training graph cache...
  Loaded 101,612 training graphs in 84.0s

  Building embedding index from training graphs...
  (This runs once per session — ~3-5 min for 100K graphs)

  Index built: torch.Size([101612, 128])  (23.9s total)

  Computing query embedding...
  Top-5 similar training ligands:
    [1] FHAAMN.xaa_6804  sim=0.9847  label=-6.73  ⚠ file not found
    [2] EHAAMN.xaa_3727  sim=0.9752  label=-7.48 kcal/mol  RMSD=113.16 Å
    [

  Viewer rendered.


In [7]:
!pip install reportlab --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.4 MB/s eta 0:00:00


In [11]:
"""
BioVanguard — Eksik Fonksiyonlar
Bu hücreyi mevcut pipeline'ınızdan SONRA çalıştırın.
generate_report, compute_molecular_profile, export_pdf ve
render_binding_site fonksiyonlarını tanımlar.

Gerekli kurulum (bir kere):
    !pip install reportlab --quiet
"""

import io
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
from datetime import datetime
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from IPython.display import display, HTML
from Bio.PDB import PDBParser

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                 Table, TableStyle, HRFlowable,
                                 Image as RLImage)
from reportlab.lib.enums import TA_CENTER

# ── Renk sabitleri (hexval() hatası olmadan) ──────────────────────────────────
COL_PASS     = "#1D9E75"
COL_FAIL     = "#E24B4A"
COL_HEADER   = "#0C447C"
COL_LIGHT    = "#F1EFE8"
COL_BORDER   = "#D3D1C7"
COL_TEXT     = "#2C2C2A"
COL_MUTED    = "#5F5E5A"
COL_ACT_BG   = "#E1F5EE"
COL_INACT_BG = "#FCEBEB"

W, H   = A4
MARGIN = 1.8 * cm

def _rl_color(hex_str):
    return colors.HexColor(hex_str)

def _pc(ok, t="Pass", f="Fail"):
    col  = COL_PASS if ok else COL_FAIL
    text = t if ok else f
    return Paragraph(
        f'<font color="{col}"><b>{text}</b></font>',
        ParagraphStyle("pc", fontSize=9, leading=12))

def _mol_png(mol, w=320, h=240):
    try:
        drawer = rdMolDraw2D.MolDraw2DCairo(w, h)
        drawer.DrawMolecule(mol)
        drawer.FinishDrawing()
        return io.BytesIO(drawer.GetDrawingText())
    except Exception:
        return None

# ── 1. compute_molecular_profile ─────────────────────────────────────────────
def compute_molecular_profile(mol, ensemble_score, xgb_pred=None, gnn_pred=None):
    mw      = Descriptors.ExactMolWt(mol)
    logp    = Descriptors.MolLogP(mol)
    hbd     = Descriptors.NumHDonors(mol)
    hba     = Descriptors.NumHAcceptors(mol)
    tpsa    = Descriptors.TPSA(mol)
    rotb    = Descriptors.NumRotatableBonds(mol)
    arom    = Descriptors.NumAromaticRings(mol)
    heavy   = Descriptors.HeavyAtomCount(mol)
    csp3    = Descriptors.FractionCSP3(mol)
    charge  = Chem.GetFormalCharge(mol)
    rings   = rdMolDescriptors.CalcNumRings(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    qed_v   = QED.qed(mol)

    lip = {
        "MW <= 500 Da": (mw   <= 500, f"{mw:.1f} Da"),
        "LogP <= 5"   : (logp <= 5,   f"{logp:.2f}"),
        "HBD <= 5"    : (hbd  <= 5,   f"{hbd}"),
        "HBA <= 10"   : (hba  <= 10,  f"{hba}"),
    }

    params  = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    params2 = FilterCatalogParams()
    params2.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)

    return {
        "formula": formula, "mw": mw, "logp": logp, "hbd": hbd, "hba": hba,
        "tpsa": tpsa, "rotb": rotb, "arom_rings": arom, "heavy_atoms": heavy,
        "frac_csp3": csp3, "formal_charge": charge, "rings_total": rings,
        "qed": qed_v,
        "lipinski"   : lip,
        "lip_passes" : sum(v[0] for v in lip.values()),
        "veber"  : (tpsa <= 140) and (rotb <= 10),
        "ghose"  : (160 <= mw <= 480) and (-0.4 <= logp <= 5.6),
        "egan"   : (tpsa <= 131.6) and (logp <= 5.88),
        "muegge" : (200 <= mw <= 600) and (-2 <= logp <= 5) and (tpsa <= 150),
        "pains_clean" : not FilterCatalog(params).HasMatch(mol),
        "brenk_clean" : not FilterCatalog(params2).HasMatch(mol),
        "bbb_permeable": tpsa < 90,
        "gi_absorption": tpsa < 140,
        "ensemble_score": ensemble_score,
        "xgb_pred"  : xgb_pred,
        "gnn_pred"  : gnn_pred,
        "is_active" : ensemble_score <= ACTIVE_THRESHOLD,
    }

# ── 2. generate_report ────────────────────────────────────────────────────────
def generate_report(mol, ensemble_score, xgb_pred=None,
                    gnn_pred=None, ligand_name=None):
    """HTML moleküler profil raporu göster. Profile dict döndürür."""
    p    = compute_molecular_profile(mol, ensemble_score, xgb_pred, gnn_pred)
    name = ligand_name or "Ligand"
    ac   = "#1D9E75" if p["is_active"] else "#E24B4A"
    ab   = "var(--color-background-success)" if p["is_active"] \
           else "var(--color-background-danger)"
    al   = "Active" if p["is_active"] else "Inactive"

    qed_pct = int(p["qed"] * 100)
    qed_col = ("#1D9E75" if p["qed"] >= 0.6
               else "#BA7517" if p["qed"] >= 0.4 else "#E24B4A")

    def pb(ok, t="Pass", f="Fail"):
        c = "#1D9E75" if ok else "#E24B4A"
        return (f'<span style="font-size:12px;font-weight:500;color:{c};">'
                f'{"✓ "+t if ok else "✗ "+f}</span>')

    lip_cards = ""
    for rule, (ok, val) in p["lipinski"].items():
        dc = "#1D9E75" if ok else "#E24B4A"
        vc = "var(--color-text-success)" if ok else "var(--color-text-danger)"
        lip_cards += f"""
        <div style="display:flex;align-items:center;gap:8px;padding:8px 10px;
                    border-radius:var(--border-radius-md);
                    border:0.5px solid var(--color-border-tertiary);">
          <div style="width:8px;height:8px;border-radius:50%;
                      background:{dc};flex-shrink:0;"></div>
          <div style="font-size:12px;color:var(--color-text-secondary);">{rule}</div>
          <div style="font-size:12px;font-weight:500;margin-left:auto;
                      color:{vc};">{val}</div>
        </div>"""

    comp = ""
    if p["gnn_pred"] is not None:
        comp += (f'<div style="font-size:12px;margin-bottom:3px;">GNN (GAT)'
                 f' &nbsp;<span style="font-weight:500;color:{ac};">'
                 f'{p["gnn_pred"]:.3f} kcal/mol</span></div>')
    if p["xgb_pred"] is not None:
        comp += (f'<div style="font-size:12px;">XGBoost'
                 f' &nbsp;<span style="font-weight:500;color:{ac};">'
                 f'{p["xgb_pred"]:.3f} kcal/mol</span></div>')

    html = f"""
<style>
.bv{{padding:1rem 0;font-size:13px;color:var(--color-text-primary)}}
.bv-hdr{{display:flex;align-items:center;gap:16px;margin-bottom:1.25rem}}
.bv-bdg{{background:var(--color-background-secondary);
  border:0.5px solid var(--color-border-tertiary);
  border-radius:var(--border-radius-md);padding:5px 12px;
  font-size:12px;color:var(--color-text-secondary)}}
.bv-act{{border-radius:var(--border-radius-md);padding:5px 12px;
  font-size:12px;font-weight:500;background:{ab};color:{ac}}}
.bv-g4{{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));
  gap:10px;margin-bottom:1.25rem}}
.bv-m{{background:var(--color-background-secondary);
  border-radius:var(--border-radius-md);padding:12px}}
.bv-ml{{font-size:11px;color:var(--color-text-secondary);margin-bottom:4px}}
.bv-mv{{font-size:20px;font-weight:500}}
.bv-mu{{font-size:11px;color:var(--color-text-tertiary);margin-left:2px}}
.bv-s{{background:var(--color-background-primary);
  border:0.5px solid var(--color-border-tertiary);
  border-radius:var(--border-radius-lg);
  padding:1rem 1.25rem;margin-bottom:1rem}}
.bv-st{{font-size:11px;font-weight:500;letter-spacing:.05em;
  color:var(--color-text-secondary);text-transform:uppercase;margin-bottom:12px}}
.bv-r{{display:flex;justify-content:space-between;align-items:center;
  padding:5px 0;border-bottom:0.5px solid var(--color-border-tertiary)}}
.bv-r:last-child{{border-bottom:none}}
.bv-rl{{color:var(--color-text-secondary)}}
.bv-rv{{font-weight:500}}
.bv-lg{{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:10px}}
.bv-af{{background:var(--color-background-secondary);
  border-radius:var(--border-radius-lg);padding:1rem 1.25rem;
  display:flex;align-items:center;justify-content:space-between}}
</style>
<div class="bv">
  <div class="bv-hdr">
    <div>
      <div style="font-size:16px;font-weight:500;margin-bottom:4px;">{name}</div>
      <div style="font-size:12px;color:var(--color-text-secondary);">
        {p["formula"]} &nbsp;·&nbsp; BioVanguard molecular profile</div>
    </div>
    <div style="margin-left:auto;display:flex;gap:8px;align-items:center;">
      <div class="bv-bdg">{p["mw"]:.1f} Da</div>
      <div class="bv-act">{al}</div>
    </div>
  </div>
  <div class="bv-g4">
    <div class="bv-m"><div class="bv-ml">Mol. weight</div>
      <div class="bv-mv">{p["mw"]:.1f}<span class="bv-mu">Da</span></div></div>
    <div class="bv-m"><div class="bv-ml">LogP</div>
      <div class="bv-mv">{p["logp"]:.2f}</div></div>
    <div class="bv-m"><div class="bv-ml">Predicted dG</div>
      <div class="bv-mv" style="color:{ac};">
        {p["ensemble_score"]:.2f}<span class="bv-mu">kcal/mol</span></div></div>
    <div class="bv-m"><div class="bv-ml">Lipinski score</div>
      <div class="bv-mv">{p["lip_passes"]}<span class="bv-mu">/ 4</span></div></div>
  </div>
  <div class="bv-s">
    <div class="bv-st">Physicochemical properties</div>
    <div class="bv-r"><span class="bv-rl">Molecular formula</span>
      <span class="bv-rv">{p["formula"]}</span></div>
    <div class="bv-r"><span class="bv-rl">Exact molecular weight</span>
      <span class="bv-rv">{p["mw"]:.4f} Da</span></div>
    <div class="bv-r"><span class="bv-rl">LogP (Crippen)</span>
      <span class="bv-rv">{p["logp"]:.3f}</span></div>
    <div class="bv-r"><span class="bv-rl">TPSA</span>
      <span class="bv-rv">{p["tpsa"]:.2f} A²</span></div>
    <div class="bv-r"><span class="bv-rl">H-bond donors</span>
      <span class="bv-rv">{p["hbd"]}</span></div>
    <div class="bv-r"><span class="bv-rl">H-bond acceptors</span>
      <span class="bv-rv">{p["hba"]}</span></div>
    <div class="bv-r"><span class="bv-rl">Rotatable bonds</span>
      <span class="bv-rv">{p["rotb"]}</span></div>
    <div class="bv-r"><span class="bv-rl">Heavy atom count</span>
      <span class="bv-rv">{p["heavy_atoms"]}</span></div>
    <div class="bv-r"><span class="bv-rl">Aromatic rings</span>
      <span class="bv-rv">{p["arom_rings"]}</span></div>
    <div class="bv-r"><span class="bv-rl">Fraction Csp3</span>
      <span class="bv-rv">{p["frac_csp3"]:.3f}</span></div>
    <div class="bv-r"><span class="bv-rl">Formal charge</span>
      <span class="bv-rv">{p["formal_charge"]:+d}</span></div>
    <div class="bv-r" style="border-bottom:none;">
      <span class="bv-rl">QED (drug-likeness)</span>
      <span class="bv-rv" style="display:flex;align-items:center;gap:8px;">
        <span style="width:80px;height:5px;
          background:var(--color-background-secondary);
          border-radius:3px;overflow:hidden;display:inline-block;">
          <span style="display:block;width:{qed_pct}%;height:100%;
            background:{qed_col};border-radius:3px;"></span>
        </span>{p["qed"]:.3f}</span></div>
  </div>
  <div class="bv-s">
    <div class="bv-st">Lipinski's rule of five</div>
    <div class="bv-lg">{lip_cards}</div>
    <div style="margin-top:10px;font-size:12px;color:var(--color-text-secondary);">
      {"Passes all 4 criteria — predicted orally bioavailable"
       if p["lip_passes"]==4
       else f"Passes {p['lip_passes']}/4 criteria — oral bioavailability may be limited"}
    </div>
  </div>
  <div class="bv-s">
    <div class="bv-st">Extended drug-likeness rules</div>
    <div class="bv-r"><span class="bv-rl">Veber (TPSA ≤ 140 Å², RotB ≤ 10)</span>
      {pb(p["veber"])}</div>
    <div class="bv-r"><span class="bv-rl">Ghose (160–480 Da, −0.4–5.6 LogP)</span>
      {pb(p["ghose"])}</div>
    <div class="bv-r"><span class="bv-rl">Egan (TPSA ≤ 131.6 Å², LogP ≤ 5.88)</span>
      {pb(p["egan"])}</div>
    <div class="bv-r" style="border-bottom:none;">
      <span class="bv-rl">Muegge (200–600 Da, −2–5 LogP)</span>
      {pb(p["muegge"])}</div>
  </div>
  <div class="bv-s">
    <div class="bv-st">ADMET &amp; structural filters</div>
    <div class="bv-r"><span class="bv-rl">PAINS filter</span>
      {pb(p["pains_clean"],"Clean","Alert detected")}</div>
    <div class="bv-r"><span class="bv-rl">Brenk structural alerts</span>
      {pb(p["brenk_clean"],"None detected","Alert detected")}</div>
    <div class="bv-r"><span class="bv-rl">BBB permeability (TPSA &lt; 90 Å²)</span>
      {pb(p["bbb_permeable"],"Likely permeable","Low permeability")}</div>
    <div class="bv-r" style="border-bottom:none;">
      <span class="bv-rl">GI absorption (TPSA &lt; 140 Å²)</span>
      {pb(p["gi_absorption"],"High","Low")}</div>
  </div>
  <div class="bv-s">
    <div class="bv-st">Predicted binding affinity</div>
    <div class="bv-af">
      <div>
        <div style="font-size:12px;color:var(--color-text-secondary);margin-bottom:4px;">
          Ensemble (GNN {int(GNN_WEIGHT*100)}% + XGBoost {int((1-GNN_WEIGHT)*100)}%)</div>
        <div style="font-size:28px;font-weight:500;color:{ac};">
          {p["ensemble_score"]:.3f} kcal/mol</div>
        <div style="font-size:11px;color:var(--color-text-tertiary);margin-top:4px;">
          Threshold: &lt;= {ACTIVE_THRESHOLD} kcal/mol &rarr; Active</div>
      </div>
      <div style="text-align:right;">
        <div style="font-size:11px;color:var(--color-text-secondary);margin-bottom:6px;">
          Component scores</div>
        {comp}
      </div>
    </div>
  </div>
</div>"""

    display(HTML(html))
    print("  Molecular profile report displayed.")
    return p   # kritik: dict döndürülüyor


# ── 3. render_binding_site ────────────────────────────────────────────────────
POLAR_RES   = {"SER","THR","CYS","TYR","ASN","GLN"}
CHARGED_POS = {"LYS","ARG","HIS"}
CHARGED_NEG = {"ASP","GLU"}

def _res_color(resname):
    r = resname.strip().upper()
    if r in CHARGED_NEG: return "#E24B4A"
    if r in CHARGED_POS: return "#378ADD"
    if r in POLAR_RES:   return "#1D9E75"
    return "#888780"

def render_binding_site(posed_mol, receptor_pdb=None, pocket_radius=12.0,
                        dpi=180, figsize=(7, 5)):
    """
    Matplotlib ile statik 3D binding site görseli.
    io.BytesIO (PNG) döndürür — PDF'e gömülebilir.
    """
    pdb_path = receptor_pdb or f"{BASE}/receptor.pdb"

    conf       = posed_mol.GetConformer()
    lig_coords = np.array([list(conf.GetAtomPosition(i))
                           for i in range(posed_mol.GetNumAtoms())])
    lig_center = lig_coords.mean(axis=0)

    parser    = PDBParser(QUIET=True)
    structure = parser.get_structure("rec", pdb_path)

    ca_pocket, sc_pocket, sc_colors = [], [], []
    all_ca = []

    for model in structure:
        for chain in model:
            for res in chain:
                ca = res.child_dict.get("CA")
                if ca is None: continue
                ca_xyz = np.array(ca.get_vector().get_array())
                all_ca.append(ca_xyz)
                dist = np.linalg.norm(ca_xyz - lig_center)
                if dist <= pocket_radius:
                    ca_pocket.append(ca_xyz)
                    sc_atoms = [a for a in res.get_atoms()
                                if a.name not in ("N","CA","C","O")
                                and a.element != "H"]
                    sc_xyz = (np.mean([a.get_vector().get_array()
                                       for a in sc_atoms], axis=0)
                              if sc_atoms else ca_xyz)
                    sc_pocket.append(sc_xyz)
                    sc_colors.append(_res_color(res.resname))

    ca_pocket = np.array(ca_pocket) if ca_pocket else np.empty((0,3))
    sc_pocket = np.array(sc_pocket) if sc_pocket else np.empty((0,3))

    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor="white")
    ax  = fig.add_subplot(111, projection='3d', facecolor="white")

    near_ca = np.array([c for c in all_ca
                        if np.linalg.norm(c-lig_center) <= pocket_radius*1.8])
    if len(near_ca) > 1:
        ax.plot(near_ca[:,0], near_ca[:,1], near_ca[:,2],
                color="#CCCCCC", lw=0.8, alpha=0.5, zorder=1)

    if len(ca_pocket) > 1:
        ax.plot(ca_pocket[:,0], ca_pocket[:,1], ca_pocket[:,2],
                color="#5F5E5A", lw=1.6, alpha=0.85, zorder=2)

    if len(sc_pocket):
        ax.scatter(sc_pocket[:,0], sc_pocket[:,1], sc_pocket[:,2],
                   c=sc_colors, s=28, alpha=0.75, zorder=3, edgecolors="none")
        for ca_xyz, sc_xyz in zip(ca_pocket, sc_pocket):
            ax.plot([ca_xyz[0],sc_xyz[0]],[ca_xyz[1],sc_xyz[1]],
                    [ca_xyz[2],sc_xyz[2]],
                    color="#AAAAAA", lw=0.6, alpha=0.6, zorder=2)

    for bond in posed_mol.GetBonds():
        i,j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ax.plot([lig_coords[i,0],lig_coords[j,0]],
                [lig_coords[i,1],lig_coords[j,1]],
                [lig_coords[i,2],lig_coords[j,2]],
                color="#1D9E75", lw=2.2, solid_capstyle='round', zorder=5)

    atom_colors = []
    for atom in posed_mol.GetAtoms():
        an = atom.GetAtomicNum()
        if an==7:    atom_colors.append("#378ADD")
        elif an==8:  atom_colors.append("#E24B4A")
        elif an==9:  atom_colors.append("#1D9E75")
        elif an==16: atom_colors.append("#BA7517")
        else:        atom_colors.append("#2C2C2A")
    ax.scatter(lig_coords[:,0], lig_coords[:,1], lig_coords[:,2],
               c=atom_colors, s=60, zorder=6,
               edgecolors="white", linewidths=0.5)

    margin = pocket_radius * 0.8
    ax.set_xlim(lig_center[0]-margin, lig_center[0]+margin)
    ax.set_ylim(lig_center[1]-margin, lig_center[1]+margin)
    ax.set_zlim(lig_center[2]-margin, lig_center[2]+margin)
    ax.set_axis_off()
    ax.view_init(elev=20, azim=-60)

    legend_elements = [
        Line2D([0],[0], color="#1D9E75", lw=2.5, label="Ligand"),
        Line2D([0],[0], color="#5F5E5A", lw=1.6, label="Pocket backbone"),
        mpatches.Patch(color="#E24B4A", label="Acidic residues"),
        mpatches.Patch(color="#378ADD", label="Basic residues"),
        mpatches.Patch(color="#1D9E75", label="Polar residues"),
        mpatches.Patch(color="#888780", label="Hydrophobic residues"),
    ]
    ax.legend(handles=legend_elements, loc="upper left",
              fontsize=6.5, framealpha=0.85,
              edgecolor="#D3D1C7", fancybox=False)

    plt.tight_layout(pad=0.3)
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=dpi, bbox_inches="tight",
                facecolor="white")
    plt.close(fig)
    buf.seek(0)
    print(f"  Binding site rendered ({len(ca_pocket)} pocket residues).")
    return buf


# ── 4. export_pdf ─────────────────────────────────────────────────────────────
def export_pdf(mol, profile, ligand_name=None, out_dir=None,
               binding_site_img=None):
    """
    Molecular profile PDF oluşturur.
    binding_site_img: render_binding_site() çıktısı (io.BytesIO) veya None
    """
    name     = ligand_name or "ligand"
    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir  = Path(out_dir or BASE)
    out_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = str(out_dir / f"{name}_report_{ts}.pdf")

    doc = SimpleDocTemplate(pdf_path, pagesize=A4,
                            leftMargin=MARGIN, rightMargin=MARGIN,
                            topMargin=MARGIN, bottomMargin=MARGIN)

    S = {
        "title" : ParagraphStyle("T",  fontSize=18, leading=22,
                                 textColor=_rl_color(COL_HEADER),
                                 spaceAfter=4, fontName="Helvetica-Bold"),
        "sub"   : ParagraphStyle("S",  fontSize=10, leading=13,
                                 textColor=_rl_color(COL_MUTED), spaceAfter=12),
        "sec"   : ParagraphStyle("SE", fontSize=11, leading=14,
                                 textColor=_rl_color(COL_HEADER),
                                 spaceBefore=14, spaceAfter=6,
                                 fontName="Helvetica-Bold"),
        "body"  : ParagraphStyle("B",  fontSize=9,  leading=13,
                                 textColor=_rl_color(COL_TEXT)),
        "label" : ParagraphStyle("L",  fontSize=9,  leading=13,
                                 textColor=_rl_color(COL_MUTED)),
        "val"   : ParagraphStyle("V",  fontSize=9,  leading=13,
                                 textColor=_rl_color(COL_TEXT),
                                 fontName="Helvetica-Bold"),
        "footer": ParagraphStyle("F",  fontSize=8,  leading=10,
                                 textColor=_rl_color(COL_MUTED),
                                 alignment=TA_CENTER),
        "cap"   : ParagraphStyle("CAP", fontSize=8, leading=11,
                                 textColor=_rl_color(COL_MUTED),
                                 alignment=TA_CENTER, spaceAfter=4),
    }

    p       = profile
    is_act  = p["is_active"]
    ac      = COL_PASS  if is_act else COL_FAIL
    act_lbl = "ACTIVE"  if is_act else "INACTIVE"
    act_bg  = _rl_color(COL_ACT_BG if is_act else COL_INACT_BG)
    ens     = p["ensemble_score"]
    avail_w = W - 2*MARGIN
    story   = []

    # Header
    story.append(Paragraph(name, S["title"]))
    story.append(Paragraph(
        f"Formula: {p['formula']}  |  BioVanguard Report  |  "
        f"{datetime.now().strftime('%d %b %Y')}", S["sub"]))
    story.append(HRFlowable(width="100%", thickness=1,
                            color=_rl_color(COL_BORDER), spaceAfter=10))

    # Summary
    cw = avail_w / 5
    story.append(Table(
        [[Paragraph("Mol. weight",S["label"]),Paragraph("LogP",S["label"]),
          Paragraph("Predicted dG",S["label"]),Paragraph("Lipinski",S["label"]),
          Paragraph("Status",S["label"])],
         [Paragraph(f"<b>{p['mw']:.1f} Da</b>",S["val"]),
          Paragraph(f"<b>{p['logp']:.2f}</b>",S["val"]),
          Paragraph(f'<font color="{ac}"><b>{ens:.2f} kcal/mol</b></font>',S["val"]),
          Paragraph(f"<b>{p['lip_passes']} / 4</b>",S["val"]),
          Paragraph(f'<font color="{ac}"><b>{act_lbl}</b></font>',S["val"])]],
        colWidths=[cw]*5,
        style=TableStyle([
            ("BACKGROUND",(0,0),(-1,0),_rl_color(COL_LIGHT)),
            ("BOX",(0,0),(-1,-1),0.5,_rl_color(COL_BORDER)),
            ("INNERGRID",(0,0),(-1,-1),0.3,_rl_color(COL_BORDER)),
            ("TOPPADDING",(0,0),(-1,-1),6),("BOTTOMPADDING",(0,0),(-1,-1),6),
            ("LEFTPADDING",(0,0),(-1,-1),8),
        ])))
    story.append(Spacer(1, 14))

    # Binding site + 2D structure
    story.append(Paragraph("Predicted Binding Pose", S["sec"]))
    if binding_site_img is not None:
        binding_site_img.seek(0)
        bs_el = RLImage(binding_site_img,
                        width=avail_w*0.62, height=avail_w*0.62*(5/7))
    else:
        bs_el = Paragraph("(binding site figure unavailable)", S["body"])

    mol_buf = _mol_png(mol, w=280, h=210)
    mol_el  = (RLImage(mol_buf, width=avail_w*0.34,
                       height=avail_w*0.34*(210/280))
               if mol_buf else Paragraph("(2D structure unavailable)", S["body"]))

    story.append(Table(
        [[bs_el, mol_el],
         [Paragraph("Predicted binding pose — receptor pocket coloured by residue type",S["cap"]),
          Paragraph("2D ligand structure",S["cap"])]],
        colWidths=[avail_w*0.63, avail_w*0.37],
        style=TableStyle([
            ("VALIGN",(0,0),(-1,-1),"TOP"),("ALIGN",(0,0),(-1,-1),"CENTER"),
            ("LEFTPADDING",(0,0),(-1,-1),0),("RIGHTPADDING",(0,0),(-1,-1),4),
            ("TOPPADDING",(0,0),(-1,-1),0),("BOTTOMPADDING",(0,0),(-1,-1),2),
        ])))
    story.append(Spacer(1, 8))

    # Physicochemical
    story.append(Paragraph("Physicochemical Properties", S["sec"]))
    props = [
        ("Formula",        p["formula"]),
        ("Exact MW",       f"{p['mw']:.4f} Da"),
        ("LogP (Crippen)", f"{p['logp']:.3f}"),
        ("TPSA",           f"{p['tpsa']:.2f} A2"),
        ("HBD / HBA",      f"{p['hbd']} / {p['hba']}"),
        ("Rotatable bonds",str(p["rotb"])),
        ("Heavy atoms",    str(p["heavy_atoms"])),
        ("Aromatic rings", str(p["arom_rings"])),
        ("Fraction Csp3",  f"{p['frac_csp3']:.3f}"),
        ("Formal charge",  f"{p['formal_charge']:+d}"),
        ("QED",            f"{p['qed']:.3f} / 1.000"),
    ]
    half = len(props)//2 + len(props)%2
    left_r, right_r = props[:half], props[half:]
    while len(right_r) < len(left_r): right_r.append(("",""))
    pw = avail_w/2 - 4
    story.append(Table(
        [[Paragraph(lk,S["label"]),Paragraph(lv,S["val"]),
          Paragraph(rk,S["label"]),Paragraph(rv,S["val"])]
         for (lk,lv),(rk,rv) in zip(left_r,right_r)],
        colWidths=[pw*0.5,pw*0.5,pw*0.5,pw*0.5],
        style=TableStyle([
            ("ROWBACKGROUNDS",(0,0),(-1,-1),[colors.white,_rl_color(COL_LIGHT)]),
            ("BOX",(0,0),(-1,-1),0.5,_rl_color(COL_BORDER)),
            ("INNERGRID",(0,0),(-1,-1),0.3,_rl_color(COL_BORDER)),
            ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
            ("LEFTPADDING",(0,0),(-1,-1),6),
        ])))

    # Lipinski
    story.append(Paragraph("Lipinski's Rule of Five", S["sec"]))
    thresh = {"MW <= 500 Da":"<= 500 Da","LogP <= 5":"<= 5",
              "HBD <= 5":"<= 5","HBA <= 10":"<= 10"}
    lip_rows = [[Paragraph("Rule",S["label"]),Paragraph("Threshold",S["label"]),
                 Paragraph("Value",S["label"]),Paragraph("Result",S["label"])]]
    for rule,(ok,val) in p["lipinski"].items():
        lip_rows.append([Paragraph(rule,S["body"]),
                         Paragraph(thresh.get(rule,"—"),S["body"]),
                         Paragraph(val,S["body"]),_pc(ok)])
    note = ("All 4 criteria satisfied — compound predicted orally bioavailable."
            if p["lip_passes"]==4 else f"{p['lip_passes']}/4 criteria satisfied.")
    lip_rows.append([Paragraph(f"<i>{note}</i>",S["body"]),"","",""])
    cw4 = avail_w/4
    story.append(Table(lip_rows,
        colWidths=[cw4*1.8,cw4*0.8,cw4*0.7,cw4*0.7],
        style=TableStyle([
            ("BACKGROUND",(0,0),(-1,0),_rl_color(COL_LIGHT)),
            ("ROWBACKGROUNDS",(0,1),(-1,-2),[colors.white,_rl_color(COL_LIGHT)]),
            ("BOX",(0,0),(-1,-1),0.5,_rl_color(COL_BORDER)),
            ("INNERGRID",(0,0),(-1,-2),0.3,_rl_color(COL_BORDER)),
            ("SPAN",(0,-1),(-1,-1)),
            ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5),
            ("LEFTPADDING",(0,0),(-1,-1),7),
        ])))

    # Drug-likeness + ADMET
    story.append(Paragraph("Extended Drug-Likeness & ADMET", S["sec"]))
    dl = [
        [Paragraph("Rule / Filter",S["label"]),
         Paragraph("Criteria",S["label"]),Paragraph("Result",S["label"])],
        [Paragraph("Veber",S["body"]),
         Paragraph("TPSA <= 140 A2, RotB <= 10",S["body"]),_pc(p["veber"])],
        [Paragraph("Ghose",S["body"]),
         Paragraph("160-480 Da, -0.4 to 5.6 LogP",S["body"]),_pc(p["ghose"])],
        [Paragraph("Egan",S["body"]),
         Paragraph("TPSA <= 131.6 A2, LogP <= 5.88",S["body"]),_pc(p["egan"])],
        [Paragraph("Muegge",S["body"]),
         Paragraph("200-600 Da, -2 to 5 LogP",S["body"]),_pc(p["muegge"])],
        [Paragraph("PAINS",S["body"]),
         Paragraph("Pan-assay interference filter",S["body"]),
         _pc(p["pains_clean"],"Clean","Alert")],
        [Paragraph("Brenk",S["body"]),
         Paragraph("Structural alerts",S["body"]),
         _pc(p["brenk_clean"],"None","Alert")],
        [Paragraph("BBB",S["body"]),
         Paragraph("TPSA < 90 A2",S["body"]),
         _pc(p["bbb_permeable"],"Likely permeable","Low")],
        [Paragraph("GI absorption",S["body"]),
         Paragraph("TPSA < 140 A2",S["body"]),
         _pc(p["gi_absorption"],"High","Low")],
    ]
    story.append(Table(dl,
        colWidths=[avail_w*0.25,avail_w*0.55,avail_w*0.20],
        style=TableStyle([
            ("BACKGROUND",(0,0),(-1,0),_rl_color(COL_LIGHT)),
            ("ROWBACKGROUNDS",(0,1),(-1,-1),[colors.white,_rl_color(COL_LIGHT)]),
            ("BOX",(0,0),(-1,-1),0.5,_rl_color(COL_BORDER)),
            ("INNERGRID",(0,0),(-1,-1),0.3,_rl_color(COL_BORDER)),
            ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5),
            ("LEFTPADDING",(0,0),(-1,-1),7),
        ])))

    # Affinity
    story.append(Paragraph("Predicted Binding Affinity", S["sec"]))
    aff = [[Paragraph("Model",S["label"]),
            Paragraph("Score (kcal/mol)",S["label"]),
            Paragraph("Weight",S["label"])]]
    if p.get("gnn_pred") is not None:
        aff.append([Paragraph("GNN (Graph Attention Network)",S["body"]),
                    Paragraph(f"<b>{p['gnn_pred']:.3f}</b>",S["val"]),
                    Paragraph(f"{int(GNN_WEIGHT*100)}%",S["body"])])
    if p.get("xgb_pred") is not None:
        aff.append([Paragraph("XGBoost",S["body"]),
                    Paragraph(f"<b>{p['xgb_pred']:.3f}</b>",S["val"]),
                    Paragraph(f"{int((1-GNN_WEIGHT)*100)}%",S["body"])])
    aff.append([Paragraph("<b>Ensemble</b>",S["val"]),
                Paragraph(f'<font color="{ac}"><b>{ens:.3f}</b></font>',S["val"]),
                Paragraph("—",S["body"])])
    aff.append([Paragraph(
        f"<i>Threshold: &lt;= {ACTIVE_THRESHOLD} kcal/mol &rarr; Active</i>",S["body"]),
        Paragraph(f'<font color="{ac}"><b>{act_lbl}</b></font>',S["val"]),
        Paragraph("",S["body"])])
    story.append(Table(aff,
        colWidths=[avail_w*0.55,avail_w*0.30,avail_w*0.15],
        style=TableStyle([
            ("BACKGROUND",(0,0),(-1,0),_rl_color(COL_LIGHT)),
            ("BACKGROUND",(0,-2),(-1,-1),act_bg),
            ("ROWBACKGROUNDS",(0,1),(-1,-3),[colors.white,_rl_color(COL_LIGHT)]),
            ("BOX",(0,0),(-1,-1),0.5,_rl_color(COL_BORDER)),
            ("INNERGRID",(0,0),(-1,-1),0.3,_rl_color(COL_BORDER)),
            ("SPAN",(0,-1),(0,-1)),
            ("TOPPADDING",(0,0),(-1,-1),6),("BOTTOMPADDING",(0,0),(-1,-1),6),
            ("LEFTPADDING",(0,0),(-1,-1),7),
        ])))

    # Footer
    story.append(Spacer(1,16))
    story.append(HRFlowable(width="100%",thickness=0.5,color=_rl_color(COL_BORDER)))
    story.append(Spacer(1,6))
    story.append(Paragraph(
        f"Generated by BioVanguard Virtual Screening Pipeline  |  "
        f"{datetime.now().strftime('%d %B %Y, %H:%M')}  |  "
        f"GNN (GAT) + XGBoost Ensemble  |  Target: C-Myc",
        S["footer"]))

    doc.build(story)
    print(f"\n  PDF saved: {pdf_path}")
    return pdf_path


# ── 5. Çalıştır ───────────────────────────────────────────────────────────────
print("All functions loaded successfully:")
print("  generate_report()      — HTML molecular profile")
print("  compute_molecular_profile() — raw property dict")
print("  render_binding_site()  — matplotlib 3D binding site PNG")
print("  export_pdf()           — full PDF report")
print()
print("Now running report + PDF export...")

# generate_report — hem gösterir hem dict döndürür
report = generate_report(
    mol            = mol,
    ensemble_score = score,
    xgb_pred       = xgb_pred,
    gnn_pred       = gnn_pred,
    ligand_name    = Path(LIGAND_SDF).stem,
)

# binding site görseli
print("\nRendering binding site figure...")
bs_img = render_binding_site(
    posed_mol    = posed_mol,
    receptor_pdb = f"{BASE}/receptor.pdb",
    pocket_radius = 12.0,
)

# PDF
print("\nExporting PDF...")
pdf_path = export_pdf(
    mol              = mol,
    profile          = report,
    ligand_name      = Path(LIGAND_SDF).stem,
    out_dir          = BASE,
    binding_site_img = bs_img,
)

from google.colab import files
files.download(pdf_path)

All functions loaded successfully:
  generate_report()      — HTML molecular profile
  compute_molecular_profile() — raw property dict
  render_binding_site()  — matplotlib 3D binding site PNG
  export_pdf()           — full PDF report

Now running report + PDF export...


  Molecular profile report displayed.

Rendering binding site figure...
  Binding site rendered (24 pocket residues).

Exporting PDF...

  PDF saved: /content/drive/MyDrive/PredictionPipeline/ZINC000002622269_report_20260411_190818.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
"""
BioVanguard — Professional Binding Site Figure
===============================================
LigPlot-tarzı 2D etkileşim görseli PDF için.
Önceki pipeline hücrelerinden sonra çalıştırın.
"""

import io
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
from Bio.PDB import PDBParser
from rdkit.Chem import Descriptors

# ── Etkileşim mesafe eşikleri ─────────────────────────────────────────────────
HBOND_CUTOFF    = 3.5   # Å — polar–polar
HYDRO_CUTOFF    = 5.0   # Å — hidrofobik kontakt
CONTACT_CUTOFF  = 4.5   # Å — genel kontakt (etiket için)
MAX_RESIDUES    = 12    # gösterilecek max residue

POLAR_ATOMS  = {7, 8, 9}      # N, O, F
HYDRO_ATOMS  = {6, 16, 17, 35, 53}  # C, S, Cl, Br, I

POLAR_RES    = {"SER","THR","CYS","TYR","ASN","GLN","HIS"}
CHARGED_POS  = {"LYS","ARG"}
CHARGED_NEG  = {"ASP","GLU"}
HYDROPHOB_RES= {"ALA","VAL","ILE","LEU","MET","PHE","TRP","PRO","GLY"}

def _res_type(resname):
    r = resname.strip().upper()
    if r in CHARGED_NEG:  return "acidic"
    if r in CHARGED_POS:  return "basic"
    if r in POLAR_RES:    return "polar"
    return "hydrophobic"

RES_COLORS = {
    "acidic"     : "#E24B4A",
    "basic"      : "#378ADD",
    "polar"      : "#1D9E75",
    "hydrophobic": "#888780",
}
RES_LABELS = {
    "acidic"     : "Acidic (Asp/Glu)",
    "basic"      : "Basic (Lys/Arg)",
    "polar"      : "Polar (Ser/Tyr/…)",
    "hydrophobic": "Hydrophobic",
}

# ── Etkileşim analizi ─────────────────────────────────────────────────────────
def analyze_interactions(posed_mol, pdb_path, cutoff=CONTACT_CUTOFF):
    """
    Ligand atomları ile protein residue'ları arasındaki
    etkileşimleri analiz eder.

    Returns
    -------
    contacts : list of dict
        Her dict: resname, resid, chain, res_center, dist_min,
                  interaction_type ('hbond'|'hydrophobic'|'contact'),
                  lig_atom_idx
    """
    conf       = posed_mol.GetConformer()
    lig_coords = np.array([list(conf.GetAtomPosition(i))
                           for i in range(posed_mol.GetNumAtoms())])
    lig_anums  = [a.GetAtomicNum() for a in posed_mol.GetAtoms()]

    parser    = PDBParser(QUIET=True)
    structure = parser.get_structure("rec", pdb_path)

    contacts = []
    for model in structure:
        for chain in model:
            for res in chain:
                if res.get_id()[0] != " ": continue  # HETATM atla

                res_heavy = [(np.array(a.get_vector().get_array()),
                              a.element.strip().upper() if a.element else 'C')
                             for a in res.get_atoms()
                             if (a.element or 'C').strip().upper() != 'H']
                if not res_heavy: continue

                # her ligand atomu ile bu residue'nun en yakın mesafesi
                best_dist   = 999
                best_lig_i  = 0
                best_res_xyz= res_heavy[0][0]

                for li, (lc, la) in enumerate(zip(lig_coords, lig_anums)):
                    for (rc, re) in res_heavy:
                        d = float(np.linalg.norm(lc - rc))
                        if d < best_dist:
                            best_dist    = d
                            best_lig_i   = li
                            best_res_xyz = rc
                            best_ra      = re
                            best_la      = la

                if best_dist > cutoff: continue

                # etkileşim tipi
                lig_polar = int(best_la) in POLAR_ATOMS
                res_polar = best_ra in ('N','O','F')

                if best_dist <= HBOND_CUTOFF and lig_polar and res_polar:
                    itype = "hbond"
                elif int(best_la) in HYDRO_ATOMS and best_ra in ('C','S'):
                    itype = "hydrophobic"
                else:
                    itype = "contact"

                res_center = np.mean([xyz for xyz,_ in res_heavy], axis=0)
                contacts.append({
                    "resname"    : res.resname,
                    "resid"      : res.get_id()[1],
                    "chain"      : chain.id,
                    "res_center" : res_center,
                    "dist_min"   : best_dist,
                    "itype"      : itype,
                    "lig_atom_i" : best_lig_i,
                    "res_type"   : _res_type(res.resname),
                })

    # mesafeye göre sırala, en yakın MAX_RESIDUES al
    contacts.sort(key=lambda x: x["dist_min"])
    return contacts[:MAX_RESIDUES]


# ── 2D projeksiyon yardımcısı ─────────────────────────────────────────────────
def _project_to_2d(coords_3d, view_elev=25, view_azim=-55):
    """Basit 3D→2D perspektif projeksiyonu."""
    el = np.radians(view_elev)
    az = np.radians(view_azim)
    # Rotation matrix
    Rx = np.array([[1,0,0],[0,np.cos(el),-np.sin(el)],[0,np.sin(el),np.cos(el)]])
    Rz = np.array([[np.cos(az),-np.sin(az),0],[np.sin(az),np.cos(az),0],[0,0,1]])
    R  = Rx @ Rz
    proj = coords_3d @ R.T
    return proj[:, :2]   # x, y alıyoruz


# ── Ana görsel fonksiyonu ─────────────────────────────────────────────────────
def render_binding_site_pro(posed_mol, receptor_pdb=None,
                            pocket_radius=12.0,
                            dpi=200, figsize=(9, 6)):
    """
    Profesyonel binding site etkileşim görseli.

    Sol panel : 3D perspektif — ligand + pocket residue'ları renkli + etiketli
    Sağ panel : etkileşim özet tablosu (H-bond / hidrofobik / kontakt)

    Returns io.BytesIO (PNG)
    """
    pdb_path = receptor_pdb or f"{BASE}/receptor.pdb"

    conf       = posed_mol.GetConformer()
    lig_coords = np.array([list(conf.GetAtomPosition(i))
                           for i in range(posed_mol.GetNumAtoms())])
    lig_center = lig_coords.mean(axis=0)

    # Etkileşimleri bul
    contacts = analyze_interactions(posed_mol, pdb_path, cutoff=CONTACT_CUTOFF)
    print(f"  Found {len(contacts)} interacting residues within {CONTACT_CUTOFF} Å")

    # 2D projeksiyon
    lig_2d = _project_to_2d(lig_coords - lig_center)
    res_2d = {}
    for c in contacts:
        rc = np.array([c["res_center"]]) - lig_center
        res_2d[c["resid"]] = _project_to_2d(rc)[0]

    # ── Figure ───────────────────────────────────────────────────────────
    fig = plt.figure(figsize=figsize, dpi=dpi, facecolor="white")
    gs  = fig.add_gridspec(1, 2, width_ratios=[1.6, 1.0],
                           left=0.02, right=0.98,
                           top=0.92, bottom=0.06, wspace=0.08)

    ax_main  = fig.add_subplot(gs[0])
    ax_table = fig.add_subplot(gs[1])
    ax_main.set_facecolor("white")
    ax_table.set_facecolor("white")

    # ── Sol panel: etkileşim diyagramı ───────────────────────────────────
    ax = ax_main

    # Ligand backbone (bağlar)
    for bond in posed_mol.GetBonds():
        i,j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ax.plot([lig_2d[i,0], lig_2d[j,0]],
                [lig_2d[i,1], lig_2d[j,1]],
                color="#2C2C2A", lw=2.0, solid_capstyle='round',
                zorder=4)

    # Ligand atomları
    atom_col = []
    for atom in posed_mol.GetAtoms():
        an = atom.GetAtomicNum()
        if an==7:    atom_col.append("#378ADD")
        elif an==8:  atom_col.append("#E24B4A")
        elif an==9:  atom_col.append("#1D9E75")
        elif an==16: atom_col.append("#BA7517")
        else:        atom_col.append("#2C2C2A")

    ax.scatter(lig_2d[:,0], lig_2d[:,1],
               c=atom_col, s=80, zorder=5,
               edgecolors="white", linewidths=0.6)

    # Ligand merkez etiketi
    ax.text(lig_2d[:,0].mean(), lig_2d[:,1].min()-0.4,
            "Ligand", ha='center', va='top',
            fontsize=8, fontweight='bold', color='#2C2C2A',
            bbox=dict(boxstyle='round,pad=0.2', fc='white',
                      ec='#D3D1C7', lw=0.5))

    # Residue'lar + etkileşim çizgileri
    plotted = set()
    for c in contacts:
        rid   = c["resid"]
        rtype = c["res_type"]
        itype = c["itype"]
        rxy   = res_2d[rid]
        rc    = RES_COLORS[rtype]

        # Etkileşim çizgisi
        lig_anchor = lig_2d[c["lig_atom_i"]]
        if itype == "hbond":
            ax.plot([lig_anchor[0], rxy[0]],
                    [lig_anchor[1], rxy[1]],
                    color="#378ADD", lw=1.4,
                    linestyle=(0,(4,2)), alpha=0.85, zorder=2)
            # mesafe etiketi
            mid = (lig_anchor + rxy) / 2
            ax.text(mid[0], mid[1],
                    f"{c['dist_min']:.1f}Å",
                    fontsize=5.5, color="#378ADD",
                    ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.1',
                              fc='white', ec='none', alpha=0.8))
        elif itype == "hydrophobic":
            ax.plot([lig_anchor[0], rxy[0]],
                    [lig_anchor[1], rxy[1]],
                    color="#888780", lw=1.0,
                    linestyle=(0,(2,3)), alpha=0.6, zorder=2)
        else:
            ax.plot([lig_anchor[0], rxy[0]],
                    [lig_anchor[1], rxy[1]],
                    color="#D3D1C7", lw=0.8,
                    linestyle=':', alpha=0.5, zorder=2)

        if rid not in plotted:
            # Residue dairesi
            circ = Circle(rxy, 0.35, color=rc, alpha=0.85, zorder=3)
            ax.add_patch(circ)
            circ2 = Circle(rxy, 0.35, fill=False,
                           edgecolor='white', lw=0.8, zorder=3)
            ax.add_patch(circ2)

            # Residue etiketi
            label = f"{c['resname']}{c['resid']}"
            # Etiketi daireden biraz uzaklaştır
            dx = rxy[0] - lig_2d[:,0].mean()
            dy = rxy[1] - lig_2d[:,1].mean()
            norm = max(np.sqrt(dx**2+dy**2), 0.01)
            lx   = rxy[0] + 0.55 * dx/norm
            ly   = rxy[1] + 0.55 * dy/norm

            txt = ax.text(lx, ly, label,
                          fontsize=7.5, fontweight='bold',
                          color=rc, ha='center', va='center',
                          zorder=6)
            txt.set_path_effects([
                pe.withStroke(linewidth=2.5, foreground='white')])
            plotted.add(rid)

    ax.set_aspect('equal')
    ax.axis('off')

    # Legend — sol panel
    legend_elements = [
        Line2D([0],[0], color='#2C2C2A', lw=2, label='Ligand bonds'),
        Line2D([0],[0], color='#378ADD', lw=1.5, ls=(0,(4,2)),
               label=f'H-bond (≤{HBOND_CUTOFF}Å)'),
        Line2D([0],[0], color='#888780', lw=1.2, ls=(0,(2,3)),
               label='Hydrophobic contact'),
        Line2D([0],[0], color='#D3D1C7', lw=1, ls=':',
               label='Van der Waals contact'),
        mpatches.Patch(color=RES_COLORS["acidic"],  label=RES_LABELS["acidic"]),
        mpatches.Patch(color=RES_COLORS["basic"],   label=RES_LABELS["basic"]),
        mpatches.Patch(color=RES_COLORS["polar"],   label=RES_LABELS["polar"]),
        mpatches.Patch(color=RES_COLORS["hydrophobic"],label=RES_LABELS["hydrophobic"]),
    ]
    ax.legend(handles=legend_elements, loc='lower left',
              fontsize=6, framealpha=0.9,
              edgecolor='#D3D1C7', fancybox=False,
              title='Interaction types', title_fontsize=6.5)

    # ── Sağ panel: etkileşim tablosu ─────────────────────────────────────
    ax2 = ax_table
    ax2.axis('off')

    # Başlık
    ax2.text(0.5, 0.98, "Interacting Residues",
             ha='center', va='top', fontsize=9, fontweight='bold',
             color='#0C447C', transform=ax2.transAxes)

    # Sütun başlıkları
    cols    = ["Residue", "Chain", "Dist (Å)", "Type"]
    col_x   = [0.02, 0.38, 0.55, 0.72]
    row_h   = 0.065
    y_start = 0.90

    for xi, col in zip(col_x, cols):
        ax2.text(xi, y_start, col,
                 ha='left', va='top', fontsize=7,
                 fontweight='bold', color='#5F5E5A',
                 transform=ax2.transAxes)

    # Ayırıcı çizgi
    ax2.axhline(y=y_start - 0.01,
            xmin=0.02, xmax=0.98,
            color='#D3D1C7', lw=0.8)

    itype_labels = {
        "hbond"      : "H-bond",
        "hydrophobic": "Hydrophob.",
        "contact"    : "VdW contact",
    }
    itype_colors = {
        "hbond"      : "#378ADD",
        "hydrophobic": "#888780",
        "contact"    : "#B4B2A9",
    }

    for ri, c in enumerate(contacts):
        y = y_start - (ri+1)*row_h - 0.015
        if y < 0.02: break

        # Satır arka planı (zebra)
        if ri % 2 == 0:
            bg = mpatches.FancyBboxPatch(
                (0.01, y-0.005), 0.97, row_h-0.005,
                boxstyle="round,pad=0.005",
                fc="#F1EFE8", ec="none", transform=ax2.transAxes,
                zorder=0)
            ax2.add_patch(bg)

        rc  = RES_COLORS[c["res_type"]]
        itc = itype_colors[c["itype"]]

        # Residue adı (renkli)
        ax2.text(col_x[0], y, f"{c['resname']}{c['resid']}",
                 ha='left', va='center', fontsize=7.5,
                 fontweight='bold', color=rc,
                 transform=ax2.transAxes)
        # Chain
        ax2.text(col_x[1], y, c["chain"],
                 ha='left', va='center', fontsize=7,
                 color='#5F5E5A', transform=ax2.transAxes)
        # Mesafe
        ax2.text(col_x[2], y, f"{c['dist_min']:.2f}",
                 ha='left', va='center', fontsize=7,
                 color='#2C2C2A', transform=ax2.transAxes)
        # Etkileşim tipi (renkli küçük badge)
        ax2.text(col_x[3], y, itype_labels[c["itype"]],
                 ha='left', va='center', fontsize=6.5,
                 color=itc, fontweight='bold',
                 transform=ax2.transAxes)

    # Özet istatistikler
    n_hb  = sum(1 for c in contacts if c["itype"]=="hbond")
    n_hy  = sum(1 for c in contacts if c["itype"]=="hydrophobic")
    n_vdw = sum(1 for c in contacts if c["itype"]=="contact")

    y_sum = max(y_start - (len(contacts)+1)*row_h - 0.04, 0.18)
    ax2.axhline(y=y_sum+0.01, xmin=0.02, xmax=0.98,
            color='#D3D1C7', lw=0.6)

    summary_lines = [
        (f"H-bonds        : {n_hb}",  "#378ADD"),
        (f"Hydrophobic    : {n_hy}",  "#888780"),
        (f"VdW contacts   : {n_vdw}", "#B4B2A9"),
        (f"Total residues : {len(contacts)}", "#2C2C2A"),
    ]
    for si, (txt, col) in enumerate(summary_lines):
        ax2.text(0.04, y_sum - si*0.055, txt,
                 ha='left', va='top', fontsize=7,
                 color=col, transform=ax2.transAxes)

    plt.suptitle("Predicted Binding Site Interactions",
                 fontsize=11, fontweight='bold', color='#0C447C',
                 y=0.98)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=dpi,
                bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    print(f"  Binding site figure rendered successfully.")
    return buf, contacts


# ── Çalıştır ──────────────────────────────────────────────────────────────────
print("Rendering professional binding site figure...")
bs_img, contacts = render_binding_site_pro(
    posed_mol    = posed_mol,
    receptor_pdb = f"{BASE}/receptor.pdb",
    pocket_radius = 12.0,
)

# PDF
print("\nExporting PDF with professional binding site figure...")
pdf_path = export_pdf(
    mol              = mol,
    profile          = report,
    ligand_name      = Path(LIGAND_SDF).stem,
    out_dir          = BASE,
    binding_site_img = bs_img,
)

from google.colab import files
files.download(pdf_path)

# Konsol özeti
print("\n" + "="*50)
print("  BINDING SITE INTERACTION SUMMARY")
print("="*50)
for c in contacts:
    print(f"  {c['resname']:<4}{c['resid']:<5} "
          f"chain {c['chain']}  "
          f"dist={c['dist_min']:.2f}Å  "
          f"[{c['itype']}]")

Rendering professional binding site figure...
  Found 11 interacting residues within 4.5 Å
  Binding site figure rendered successfully.

Exporting PDF with professional binding site figure...

  PDF saved: /content/drive/MyDrive/PredictionPipeline/ZINC000002622269_report_20260411_191106.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  BINDING SITE INTERACTION SUMMARY
  ALA 167   chain B  dist=0.85Å  [hydrophobic]
  ASN 166   chain B  dist=2.13Å  [contact]
  LEU 168   chain B  dist=2.44Å  [contact]
  ARG 80    chain A  dist=2.58Å  [contact]
  LYS 163   chain B  dist=2.71Å  [hbond]
  ARG 164   chain B  dist=3.05Å  [contact]
  LEU 76    chain A  dist=3.11Å  [hydrophobic]
  GLU 170   chain B  dist=3.68Å  [contact]
  LEU 77    chain A  dist=3.84Å  [contact]
  GLN 165   chain B  dist=4.29Å  [contact]
  LEU 84    chain A  dist=4.41Å  [hydrophobic]
